In [ ]:
import numpy as np
from votekit.cvr_loaders import load_scottish
import matplotlib.pyplot as plt
from votekit import PreferenceProfile, PreferenceInterval
from votekit import Ballot
import votekit.ballot_generator as bg
from votekit.elections import STV, fractional_transfer
from votekit.plots import plot_summary_stats
import math
from fractions import Fraction
import itertools as it
import pickle

# used to import local files
import sys  
sys.path.insert(1, './')

from swap_distance import *

from collections import Counter
from scipy.stats import wasserstein_distance, kstest
import pandas as pd

plt.rcParams['text.usetex'] = True
SMALL_SIZE = 16
MEDIUM_SIZE = 20
BIGGER_SIZE = 24

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
b_bloc_parties = ['Scottish National Party (SNP)', 'Green (Gr)']
file_names = {("fife", 2022, 21): "../election_data/fife_2022_ward21.csv",
              ("aberdeen", 2017, 12) : "../election_data/aberdeen_2017_ward12.csv",
              ("aberdeen", 2022, 12): "../election_data/aberdeen_2022_ward12.csv",
              ("angus", 2012, 8): "../election_data/angus_2012_ward8.csv",
              ("falkirk", 2017, 6): "../election_data/falkirk_2017_ward6.csv",
              ("clackmannanshire", 2012, 2): "../election_data/clackmannanshire_2012_ward2.csv",
              ("renfrewshire", 2017, 1): "../election_data/renfrewshire_2017_ward1.csv",
              ("glasgow", 2012, 16): "../election_data/glasgow_2012_ward16.csv",
              ("north-ayrshire", 2022, 1): "../election_data/north_ayrshire_2022_north_coast.csv"
              }


new_models = ["CS-C", "CS-W", "n-BT", "n-PL", "s-BT", "s-PL"]
old_models = ["SB", "IC", "IAC"]
scottish_color = ["#1560BD"]
model_to_color = {'CS-C': '#D2691E', 'CS-W': '#E32636', 'n-BT': '#8B008B', 'n-PL': '#FFB7C5', 's-BT': '#FFBF00', 's-PL': '#8DB600',"SB":"#09FCEE", "IC":"#1A7601",
                      "IAC":"#929292" }

# Histograms of Scottish profiles

In [ ]:
for key, file_name in file_names.items():
    city, year, ward = key
    label = f"{city.capitalize()} Ward {ward} {year}"
    print(label)
    scottish_profile, num_seats, cand_list, cand_to_party, ward = load_scottish(file_name)
    cand_to_bloc = {c:"B" if cand_to_party[c] in b_bloc_parties 
                else "A" for c in cand_list}
    
    # count the number of candidates in each bloc
    bloc_to_cand_num = {"A": len([c for c, bloc in cand_to_bloc.items() if bloc == "A"]),
                        "B": len([c for c, bloc in cand_to_bloc.items() if bloc == "B"])}
    
    bloc_to_cand_num = {"A": len([c for c, bloc in cand_to_bloc.items() if bloc == "A"]),
                        "B": len([c for c, bloc in cand_to_bloc.items() if bloc == "B"])}
    
    
    dist_to_solid = dist_profile_to_solid(scottish_profile, cand_to_bloc, bloc_to_cand_num)

    fig,ax = plt.subplots(figsize=(8,6))
    bin_min = 0
    bin_max = max(dist_to_solid)
    bins = np.arange(bin_min-.5, bin_max+1.5, 1)
    ax.hist(dist_to_solid, bins = bins, label = label , density= True, alpha = 1, color = scottish_color)
    ax.set_xticks(range(int(bin_max)+1))
    ax.tick_params(axis='x', labelrotation=90)
    # ax.legend()
    ax.set_xlabel("Distance to solid (A over B)")
    ax.set_ylabel("Density")
    ax.set_title(f"{label}")

    # plt.show()
    plt.tight_layout()
    plt.savefig(f"figures/histograms/{label}_distance_to_solid_hist.pdf")
    plt.clf()
    

